# Preprocessing – Titanic – Bouckaert John, Hugo Fievet

## Introduction

Dans ce notebook, nous réalisons le prétraitement du dataset Titanic en vue des étapes suivantes de sélection de variables et de modélisation.

L’analyse exploratoire complète a déjà été réalisée dans la phase 1. Nous ne répétons donc pas ici l’étude descriptive du dataset. Nous nous appuyons directement sur les constats précédents pour préparer les données : gestion des valeurs manquantes, traitement des variables catégorielles, sélection des colonnes utiles et préparation des jeux d’entraînement et de test.

L’objectif est d’obtenir des données propres, cohérentes et exploitables pour la suite du travail, puis de sauvegarder les fichiers CSV nécessaires.

## Table of contents

1. [Initial separation between features and target](#1-initial-separation-between-features-and-target)  
2. [Feature engineering based on the conclusions of phase 1](#2-feature-engineering-based-on-the-conclusions-of-phase-1)  
3. [Removing raw variables after feature engineering](#3-removing-raw-variables-after-feature-engineering)  
4. [Train-test split before preprocessing](#4-train-test-split-before-preprocessing)  
5. [Identifying missing values in the training and test sets](#5-identifying-missing-values-in-the-training-and-test-sets)  
6. [Imputing missing values](#6-imputing-missing-values)  
7. [Encoding categorical variables](#7-encoding-categorical-variables)  
8. [Scaling numerical features](#8-scaling-numerical-features)  
9. [Resetting indexes and saving the preprocessed datasets](#9-resetting-indexes-and-saving-the-preprocessed-datasets)  
10. [Conclusion](#conclusion)

## 1. Initial separation between features and target

Dans cette étape, nous chargeons le dataset et nous séparons la variable cible des variables explicatives.  
L’objectif est de distinguer clairement les données à prédire (`survived`) des variables qui serviront à construire les modèles.

Cette séparation constitue une base essentielle pour le prétraitement, car les transformations ultérieures devront être appliquées uniquement aux variables explicatives.

In [2]:
import pandas as pd

# Chargement du dataset
df = pd.read_csv("Titanic Dataset.csv")

# Variable cible
target = "survived"

# Séparation initiale
X = df.drop(columns=[target])
y = df[target]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print("\nColonnes présentes dans X :")
print(X.columns.tolist())

Dimensions de X : (1309, 10)
Dimensions de y : (1309,)

Colonnes présentes dans X :
['pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


La séparation entre les variables explicatives et la variable cible a bien été réalisée.  
Le jeu de données explicatives `X` contient **1309 observations** et **10 variables**, tandis que la variable cible `y` contient **1309 valeurs**.

Les variables explicatives conservées à ce stade sont : `pclass`, `name`, `sex`, `age`, `sibsp`, `parch`, `ticket`, `fare`, `cabin` et `embarked`.  
Cette étape confirme que la variable `survived` a bien été isolée et que le prétraitement pourra désormais se concentrer uniquement sur les variables d’entrée.

## 2. Feature engineering based on the conclusions of phase 1

Dans cette étape, nous ne supprimons pas directement certaines variables brutes.  
Nous commençons par exploiter les informations intéressantes mises en évidence dans la phase 1.

Plus précisément, nous créons de nouvelles variables à partir de colonnes qui ne sont pas directement utilisables sous leur forme initiale. Le nom du passager permet par exemple d’extraire un titre, tandis que la variable `cabin` permet d’indiquer si une information de cabine est disponible ou non.

L’objectif est de conserver l’information pertinente révélée par l’analyse exploratoire tout en préparant un jeu de données plus exploitable pour la modélisation.

In [ ]:
# Copie de travail
X_preprocessed = X.copy()

# 1. Extraction du titre à partir du nom
X_preprocessed["title"] = X_preprocessed["name"].str.extract(r",\s*([^\.]+)\.", expand=False)

# 2. Regroupement des titres
common_titles = ["Mr", "Mrs", "Miss", "Master"]
X_preprocessed["title_grouped"] = X_preprocessed["title"].apply(
    lambda x: x if x in common_titles else "Other"
)

# 3. Création d'une variable indiquant si la cabine est renseignée
X_preprocessed["has_cabin"] = X_preprocessed["cabin"].notna().astype(int)

# 4. Création de la taille de la famille
X_preprocessed["family_size"] = X_preprocessed["sibsp"] + X_preprocessed["parch"] + 1

# 5. Regroupement de la taille de la famille
def group_family_size(size):
    if size == 1:
        return "alone"
    elif 2 <= size <= 4:
        return "small_family"
    else:
        return "large_family"

X_preprocessed["family_group"] = X_preprocessed["family_size"].apply(group_family_size)

# 6. Aperçu des nouvelles variables
print("Nouvelles colonnes créées :")
print(["title", "title_grouped", "has_cabin", "family_size", "family_group"])

print("\nPremières lignes des variables créées :")
print(X_preprocessed[["name", "title", "title_grouped", "cabin", "has_cabin", "sibsp", "parch", "family_size", "family_group"]].head())

print("\nRépartition de title_grouped :")
print(X_preprocessed["title_grouped"].value_counts(dropna=False))

print("\nRépartition de has_cabin :")
print(X_preprocessed["has_cabin"].value_counts(dropna=False))

print("\nRépartition de family_size :")
print(X_preprocessed["family_size"].value_counts().sort_index())

print("\nRépartition de family_group :")
print(X_preprocessed["family_group"].value_counts(dropna=False))

Nouvelles colonnes créées :
['title', 'title_grouped', 'has_cabin', 'family_size', 'family_group']

Premières lignes des variables créées :
                                                 name   title title_grouped  \
0                       Allen, Miss. Elisabeth Walton    Miss          Miss   
1                      Allison, Master. Hudson Trevor  Master        Master   
2                        Allison, Miss. Helen Loraine    Miss          Miss   
3                Allison, Mr. Hudson Joshua Creighton      Mr            Mr   
4     Allison, Mrs. Hudson J C (Bessie Waldo Daniels)     Mrs           Mrs   
..                                                ...     ...           ...   
95              Dodge, Mrs. Washington (Ruth Vidaver)     Mrs           Mrs   
96                         Douglas, Mr. Walter Donald      Mr            Mr   
97  Douglas, Mrs. Frederick Charles (Mary Helene B...     Mrs           Mrs   
98        Douglas, Mrs. Walter Donald (Mahala Dutton)     Mrs         

Les nouvelles variables créées permettent de mieux traduire certaines informations mises en évidence dans la phase 1 tout en rendant le dataset plus exploitable pour la modélisation.

L’extraction du titre à partir de `name` confirme que cette information est structurée et exploitable. La majorité des passagers appartiennent aux groupes `Mr`, `Miss`, `Mrs` et `Master`, tandis qu’un petit nombre de titres plus rares a été regroupé dans la catégorie `Other`. Ce regroupement permet de simplifier la variable tout en conservant une information sociale potentiellement liée à la survie.

La variable `has_cabin` transforme également une colonne difficile à utiliser directement en une information binaire simple. Même si `cabin` contient beaucoup de valeurs manquantes, le fait qu’une cabine soit renseignée ou non peut constituer un signal pertinent, notamment en lien avec la classe ou le niveau de confort du passager.

La variable `family_size` résume les colonnes `sibsp` et `parch` en une mesure synthétique de la taille de la famille. On observe que la majorité des passagers voyagent seuls, mais aussi qu’il existe plusieurs tailles de familles beaucoup moins fréquentes.

Afin de mieux tenir compte de la relation non linéaire observée entre la taille de la famille et la survie dans la phase 1, nous avons créé une nouvelle variable catégorielle `family_group`. Cette variable distingue trois profils : les passagers voyageant seuls (`alone`), ceux voyageant en petite famille (`small_family`, de 2 à 4 personnes) et ceux appartenant à une grande famille (`large_family`, plus de 4 personnes). Cette représentation est plus adaptée que la seule variable numérique `family_size`, car elle permet de mieux refléter les différences de comportement observées entre ces groupes.

## 3. Removing raw variables after feature engineering

Dans cette étape, nous supprimons certaines variables brutes dont l’information utile a déjà été extraite sous une forme plus exploitable.

L’objectif est de conserver les variables transformées pertinentes issues du feature engineering, tout en retirant les colonnes initiales devenues redondantes ou peu adaptées à la modélisation.

In [12]:
# Colonnes brutes supprimées après feature engineering
columns_to_drop = ["name", "title", "ticket", "cabin"]

# Suppression des colonnes
X_preprocessed = X_preprocessed.drop(columns=columns_to_drop, errors="ignore")

print("Colonnes supprimées :")
print(columns_to_drop)

print("\nColonnes restantes :")
print(X_preprocessed.columns.tolist())

print("\nDimensions après suppression des colonnes :")
print(X_preprocessed.shape)

print("\nPremières lignes après suppression des colonnes :")
print(X_preprocessed.head())

Colonnes supprimées :
['name', 'title', 'ticket', 'cabin']

Colonnes restantes :
['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'title_grouped', 'has_cabin', 'family_size', 'family_group']

Dimensions après suppression des colonnes :
(1309, 11)

Premières lignes après suppression des colonnes :
   pclass     sex    age  sibsp  parch      fare embarked title_grouped  \
0       1  female  29.00      0      0  211.3375        S          Miss   
1       1    male   0.92      1      2  151.5500        S        Master   
2       1  female   2.00      1      2  151.5500        S          Miss   
3       1    male  30.00      1      2  151.5500        S            Mr   
4       1  female  25.00      1      2  151.5500        S           Mrs   

   has_cabin  family_size  family_group  
0          1            1         alone  
1          1            4  small_family  
2          1            4  small_family  
3          1            4  small_family  
4          1            4  

Les variables brutes `name`, `ticket` et `cabin` ont été supprimées, car elles étaient soit difficilement exploitables sous leur forme initiale, soit déjà partiellement remplacées par des variables dérivées plus pertinentes. La variable intermédiaire `title` a également été retirée, puisque seule la version regroupée `title_grouped` a été conservée pour la suite.

Après cette étape, le jeu de données contient encore **1309 observations** mais désormais **11 variables explicatives**. Cette structure est plus adaptée au prétraitement, car elle conserve à la fois des variables initiales encore exploitables (`pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked`) et des variables construites à partir de l’analyse exploratoire (`title_grouped`, `has_cabin`, `family_size`, `family_group`).

L’introduction de `family_group` enrichit la représentation de la structure familiale. Contrairement à `family_size`, qui résume cette information sous forme numérique, `family_group` distingue des profils plus interprétables et plus proches des constats de la phase 1. Cette étape permet donc d’obtenir un jeu de données plus cohérent pour les traitements suivants.

## 4. Train-test split 

Dans cette étape, nous séparons les données en un jeu d’entraînement et un jeu de test avant d’appliquer les traitements de prétraitement.

Cette démarche permet d’éviter les fuites d’information : les choix de prétraitement, comme l’imputation ou la standardisation, devront être appris sur les données d’entraînement puis appliqués au jeu de test.

In [13]:
from sklearn.model_selection import train_test_split

# Séparation en jeu d'entraînement et jeu de test
X_train, X_test, y_train, y_test = train_test_split(
    X_preprocessed,
    y,
    test_size=0.2,
    random_state=42, # Pour que le découpage soit reproductible. Si on relance le notebook, on obtient le même split.
    stratify=y
)

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)
print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test :", y_test.shape)

print("\nRépartition de y_train (%) :")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nRépartition de y_test (%) :")
print((y_test.value_counts(normalize=True) * 100).round(2))

Dimensions de X_train : (1047, 11)
Dimensions de X_test : (262, 11)
Dimensions de y_train : (1047,)
Dimensions de y_test : (262,)

Répartition de y_train (%) :
survived
0    61.8
1    38.2
Name: proportion, dtype: float64

Répartition de y_test (%) :
survived
0    61.83
1    38.17
Name: proportion, dtype: float64


La séparation entre le jeu d’entraînement et le jeu de test a bien été réalisée. Le dataset contient désormais **1047 observations** dans `X_train` et **262 observations** dans `X_test`, ce qui correspond à une répartition classique de **80 % pour l’entraînement** et **20 % pour le test**.

L’utilisation de l’argument `stratify=y` permet de conserver une répartition très proche de la variable cible dans les deux sous-ensembles. En effet, la proportion de passagers non survivants est d’environ **61.8 %** dans le jeu d’entraînement et **61.83 %** dans le jeu de test, tandis que la proportion de survivants reste proche de **38.2 %** dans les deux cas.

Cette étape est importante car elle garantit que les futurs traitements de prétraitement seront appris uniquement sur les données d’entraînement. Cela limite les fuites d’information et permet d’évaluer plus correctement les performances des modèles sur des données non vues.

## 5. Identifying missing values in the training and test sets

Dans cette étape, nous examinons la répartition des valeurs manquantes dans les jeux d’entraînement et de test après la séparation des données.

L’objectif est de vérifier quelles variables nécessitent une imputation et de s’assurer que les traitements seront ensuite définis à partir du jeu d’entraînement, puis appliqués de manière cohérente au jeu de test.

In [14]:
print("Valeurs manquantes dans X_train :")
print(X_train.isna().sum())

print("\nProportion de valeurs manquantes dans X_train (%) :")
print((X_train.isna().mean() * 100).round(2))

print("\nValeurs manquantes dans X_test :")
print(X_test.isna().sum())

print("\nProportion de valeurs manquantes dans X_test (%) :")
print((X_test.isna().mean() * 100).round(2))

Valeurs manquantes dans X_train :
pclass             0
sex                0
age              209
sibsp              0
parch              0
fare               1
embarked           0
title_grouped      0
has_cabin          0
family_size        0
family_group       0
dtype: int64

Proportion de valeurs manquantes dans X_train (%) :
pclass            0.00
sex               0.00
age              19.96
sibsp             0.00
parch             0.00
fare              0.10
embarked          0.00
title_grouped     0.00
has_cabin         0.00
family_size       0.00
family_group      0.00
dtype: float64

Valeurs manquantes dans X_test :
pclass            0
sex               0
age              54
sibsp             0
parch             0
fare              0
embarked          2
title_grouped     0
has_cabin         0
family_size       0
family_group      0
dtype: int64

Proportion de valeurs manquantes dans X_test (%) :
pclass            0.00
sex               0.00
age              20.61
sibsp        

L’analyse des valeurs manquantes après le train-test split montre que la situation est désormais plus ciblée. Dans le jeu d’entraînement, seule la variable `age` présente un nombre important de valeurs manquantes, avec **209 observations** concernées, soit environ **19.96 %** du jeu. La variable `fare` ne contient qu’une seule valeur manquante, tandis que toutes les autres colonnes sont complètes.

Dans le jeu de test, on retrouve une situation très proche pour `age`, avec **54 valeurs manquantes** soit environ **20.61 %**. La variable `embarked` contient quant à elle **2 valeurs manquantes**, ce qui reste très limité. Aucune autre variable ne présente de données manquantes.

Ces résultats confirment que le prétraitement devra principalement porter sur l’imputation de `age`, ainsi que sur un traitement ponctuel de `fare` et `embarked`. Le fait d’observer des proportions similaires de valeurs manquantes pour `age` dans les jeux d’entraînement et de test montre que cette difficulté est structurelle dans le dataset et non liée au découpage des données.

## 6. Imputing missing values

Dans cette étape, nous traitons les valeurs manquantes restantes dans les jeux d’entraînement et de test.

Compte tenu de la nature des variables concernées, nous n’utilisons pas une stratégie unique pour toutes les colonnes. Pour la variable `age`, nous privilégions une imputation par groupes afin de conserver une structure plus réaliste des données. Pour `fare` et `embarked`, nous utilisons des méthodes plus simples adaptées au très faible nombre de valeurs manquantes.

In [15]:
# Médianes d'âge calculées sur le jeu d'entraînement par groupe
age_medians_by_group = X_train.groupby(["title_grouped", "pclass"])["age"].median()

# Médianes de secours calculées sur le jeu d'entraînement par titre
age_medians_by_title = X_train.groupby("title_grouped")["age"].median()

# Médiane globale de secours
age_median_global = X_train["age"].median()

# Valeurs d'imputation pour les autres variables
fare_median = X_train["fare"].median()
embarked_mode = X_train["embarked"].mode()[0]

print("Médianes d'âge par groupe (title_grouped, pclass) :")
print(age_medians_by_group)

print("\nMédianes d'âge par titre :")
print(age_medians_by_title)

print("\nMédiane globale de l'âge :", age_median_global)
print("Médiane du tarif utilisée pour l'imputation :", fare_median)
print("Mode du port d'embarquement utilisé pour l'imputation :", embarked_mode)


def impute_age(row):
    if pd.notna(row["age"]):
        return row["age"]

    group_key = (row["title_grouped"], row["pclass"])

    if group_key in age_medians_by_group.index and pd.notna(age_medians_by_group.loc[group_key]):
        return age_medians_by_group.loc[group_key]

    if row["title_grouped"] in age_medians_by_title.index and pd.notna(age_medians_by_title.loc[row["title_grouped"]]):
        return age_medians_by_title.loc[row["title_grouped"]]

    return age_median_global


# Imputation de l'âge
X_train["age"] = X_train.apply(impute_age, axis=1)
X_test["age"] = X_test.apply(impute_age, axis=1)

# Imputation de fare et embarked à partir du train
X_train["fare"] = X_train["fare"].fillna(fare_median)
X_test["fare"] = X_test["fare"].fillna(fare_median)

X_train["embarked"] = X_train["embarked"].fillna(embarked_mode)
X_test["embarked"] = X_test["embarked"].fillna(embarked_mode)

# Vérification finale
print("\nValeurs manquantes dans X_train après imputation :")
print(X_train.isna().sum())

print("\nValeurs manquantes dans X_test après imputation :")
print(X_test.isna().sum())

Médianes d'âge par groupe (title_grouped, pclass) :
title_grouped  pclass
Master         1          5.0
               2          1.5
               3          5.5
Miss           1         30.0
               2         19.0
               3         18.0
Mr             1         41.0
               2         30.0
               3         26.0
Mrs            1         45.0
               2         31.5
               3         31.0
Other          1         44.5
               2         42.0
               3          NaN
Name: age, dtype: float64

Médianes d'âge par titre :
title_grouped
Master     4.0
Miss      21.0
Mr        29.0
Mrs       35.5
Other     44.0
Name: age, dtype: float64

Médiane globale de l'âge : 28.0
Médiane du tarif utilisée pour l'imputation : 13.81665
Mode du port d'embarquement utilisé pour l'imputation : S

Valeurs manquantes dans X_train après imputation :
pclass           0
sex              0
age              0
sibsp            0
parch            0
fare          

L’imputation des valeurs manquantes a été réalisée en tenant compte de la structure du dataset. Plutôt que d’utiliser une médiane globale unique pour `age`, nous avons choisi une stratégie plus fine basée sur les groupes `title_grouped` et `pclass`, ce qui permet d’attribuer des âges plus cohérents avec le profil des passagers. Par exemple, les valeurs imputées pour `Master` restent nettement plus faibles que celles de `Mr` ou `Mrs`, ce qui correspond mieux à la réalité observée dans les données.

Lorsque la médiane d’un groupe précis n’était pas disponible, une médiane par titre a été utilisée comme solution de secours, puis la médiane globale uniquement en dernier recours. Cette hiérarchie permet de limiter la perte d’information et d’éviter une uniformisation excessive de la variable `age`.

Pour `fare` et `embarked`, le nombre de valeurs manquantes étant très faible, une imputation simple a été jugée suffisante : la médiane pour `fare` et le mode pour `embarked`, calculés uniquement à partir du jeu d’entraînement.

Après cette étape, aucune valeur manquante ne subsiste ni dans `X_train` ni dans `X_test`. Le dataset est donc désormais complet et prêt pour les étapes suivantes du prétraitement, notamment l’encodage des variables catégorielles et la standardisation éventuelle des variables numériques.

## 7. Encoding categorical variables

Dans cette étape, nous transformons les variables catégorielles restantes en variables numériques afin de rendre le dataset compatible avec les algorithmes de machine learning.

Nous appliquons l’encodage sur le jeu d’entraînement, puis nous reproduisons la même structure sur le jeu de test afin de conserver des colonnes cohérentes dans les deux sous-ensembles.

In [16]:
# Encodage des variables catégorielles
categorical_columns = ["pclass", "sex", "embarked", "title_grouped", "family_group"]

X_train_encoded = pd.get_dummies(X_train, columns=categorical_columns, drop_first=False, dtype=int)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_columns, drop_first=False, dtype=int)

# Alignement des colonnes entre train et test
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join="left", axis=1, fill_value=0)

print("Colonnes catégorielles encodées :")
print(categorical_columns)

print("\nDimensions de X_train après encodage :", X_train_encoded.shape)
print("Dimensions de X_test après encodage :", X_test_encoded.shape)

print("\nColonnes après encodage :")
print(X_train_encoded.columns.tolist())

print("\nAperçu de X_train après encodage :")
print(X_train_encoded.head())

Colonnes catégorielles encodées :
['pclass', 'sex', 'embarked', 'title_grouped', 'family_group']

Dimensions de X_train après encodage : (1047, 22)
Dimensions de X_test après encodage : (262, 22)

Colonnes après encodage :
['age', 'sibsp', 'parch', 'fare', 'has_cabin', 'family_size', 'pclass_1', 'pclass_2', 'pclass_3', 'sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'title_grouped_Master', 'title_grouped_Miss', 'title_grouped_Mr', 'title_grouped_Mrs', 'title_grouped_Other', 'family_group_alone', 'family_group_large_family', 'family_group_small_family']

Aperçu de X_train après encodage :
       age  sibsp  parch     fare  has_cabin  family_size  pclass_1  pclass_2  \
999   18.0      0      0   7.7500          0            1         0         0   
392   24.0      1      0  27.7208          0            2         0         1   
628   11.0      4      2  31.2750          0            7         0         0   
1165  25.0      0      0   7.2250          0            1    

L’encodage des variables catégorielles a permis de transformer `pclass`, `sex`, `embarked`, `title_grouped` et `family_group` en variables numériques exploitables par les futurs modèles. Cette étape était nécessaire, car les algorithmes de machine learning utilisés dans ce travail ne peuvent pas traiter directement des catégories textuelles.

Après cette transformation, le jeu d’entraînement et le jeu de test contiennent chacun **22 variables**, ce qui montre que la structure a bien été harmonisée entre les deux sous-ensembles. Cette cohérence est essentielle pour garantir que les modèles seront ensuite entraînés et évalués sur les mêmes colonnes.

L’aperçu obtenu montre que chaque modalité est désormais représentée par une colonne dédiée, comme `pclass_1`, `sex_female`, `embarked_S` ou encore `title_grouped_Mr`. La variable `family_group` est également représentée par trois colonnes binaires : `family_group_alone`, `family_group_large_family` et `family_group_small_family`. Cette représentation est plus fidèle aux constats de la phase 1, car elle distingue des profils familiaux différents au lieu de résumer cette information à une seule variable numérique.

Le dataset est maintenant presque entièrement prêt pour la modélisation. Il reste encore à standardiser les variables quantitatives, afin d’éviter qu’une variable à forte amplitude domine artificiellement les autres, en particulier dans les modèles sensibles aux distances comme le KNN.

## 8. Scaling numerical features

Dans cette étape, nous standardisons les variables quantitatives du dataset.  
Cette transformation est particulièrement importante pour les modèles sensibles aux distances, comme le KNN, car elle évite qu’une variable ayant une grande amplitude domine artificiellement les autres.

Les variables issues de l’encodage catégoriel sont déjà représentées en `0` et `1`, elles ne nécessitent donc pas de standardisation.

In [17]:
from sklearn.preprocessing import StandardScaler

# Colonnes quantitatives à standardiser
columns_to_scale = ["age", "fare", "sibsp", "parch", "family_size"]

scaler = StandardScaler()

# Standardisation à partir du jeu d'entraînement
X_train_encoded[columns_to_scale] = scaler.fit_transform(X_train_encoded[columns_to_scale])
#fit_transform sur le train, le scaler apprend: la moyenne et l’écart-type a partir de X_train seulement
#transform sur le test: ensuite, il applique ces mêmes paramètres à X_test.
X_test_encoded[columns_to_scale] = scaler.transform(X_test_encoded[columns_to_scale])

print("Colonnes standardisées :")
print(columns_to_scale)

print("\nAperçu de X_train après standardisation :")
print(X_train_encoded.head())

print("\nTypes de variables après standardisation :")
print(X_train_encoded.dtypes)

Colonnes standardisées :
['age', 'fare', 'sibsp', 'parch', 'family_size']

Aperçu de X_train après standardisation :
           age     sibsp     parch      fare  has_cabin  family_size  \
999  -0.820265 -0.479571 -0.447597 -0.499246          0    -0.557940   
392  -0.371539  0.510785 -0.447597 -0.090614          0     0.083293   
628  -1.343779  3.481854  1.872374 -0.017890          0     3.289456   
1165 -0.296752 -0.479571 -0.447597 -0.509988          0    -0.557940   
604  -0.969840 -0.479571 -0.447597 -0.501292          0    -0.557940   

      pclass_1  pclass_2  pclass_3  sex_female  ...  embarked_Q  embarked_S  \
999          0         0         1           1  ...           1           0   
392          0         1         0           1  ...           0           0   
628          0         0         1           1  ...           0           1   
1165         0         0         1           0  ...           0           0   
604          0         0         1           1  ...    

La standardisation a été appliquée aux variables quantitatives `age`, `fare`, `sibsp`, `parch` et `family_size`. Cette étape était nécessaire car ces variables n’étaient pas exprimées sur la même échelle. Sans cette transformation, une variable comme `fare` aurait pu influencer de manière disproportionnée les calculs de distance, ce qui aurait été problématique en particulier pour le modèle KNN.

L’aperçu de `X_train` confirme que ces variables quantitatives ont bien été transformées : elles prennent désormais des valeurs positives ou négatives, ce qui est normal après une standardisation. Ces valeurs ne correspondent plus aux unités d’origine, mais à la position relative de chaque observation par rapport à la moyenne du jeu d’entraînement.

Les autres variables, comme `has_cabin`, `pclass_1`, `sex_female`, `title_grouped_Mr` ou encore les colonnes issues de `family_group`, ont été conservées sous forme binaire (`0` ou `1`). Elles sont déjà numériques et ne nécessitaient pas de standardisation supplémentaire.

Enfin, les types de variables obtenus confirment que le dataset est maintenant entièrement numérique. Les variables quantitatives standardisées apparaissent en `float64`, tandis que les variables binaires issues de l’encodage restent en `int64`. Le jeu de données est donc désormais prêt pour l’export final en fichiers CSV et pour la suite du travail consacrée à la sélection des variables.

## 9. Resetting indexes and saving the preprocessed datasets

Dans cette dernière étape, nous remettons les index à zéro afin d’obtenir des fichiers plus lisibles et plus propres pour la suite du travail.

Nous sauvegardons ensuite les jeux de données prétraités dans des fichiers CSV distincts pour les variables explicatives et la variable cible, en séparant bien les données d’entraînement et de test.

In [18]:
# Remise à zéro des index
X_train_encoded = X_train_encoded.reset_index(drop=True)
X_test_encoded = X_test_encoded.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("Dimensions finales :")
print("X_train :", X_train_encoded.shape)
print("X_test  :", X_test_encoded.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

print("\nAperçu de X_train avec index réinitialisé :")
print(X_train_encoded.head())

# Sauvegarde des fichiers CSV
X_train_encoded.to_csv("xtrain.csv", index=False)
X_test_encoded.to_csv("xtest.csv", index=False)
y_train.to_csv("ytrain.csv", index=False)
y_test.to_csv("ytest.csv", index=False)

Dimensions finales :
X_train : (1047, 22)
X_test  : (262, 22)
y_train : (1047,)
y_test  : (262,)

Aperçu de X_train avec index réinitialisé :
        age     sibsp     parch      fare  has_cabin  family_size  pclass_1  \
0 -0.820265 -0.479571 -0.447597 -0.499246          0    -0.557940         0   
1 -0.371539  0.510785 -0.447597 -0.090614          0     0.083293         0   
2 -1.343779  3.481854  1.872374 -0.017890          0     3.289456         0   
3 -0.296752 -0.479571 -0.447597 -0.509988          0    -0.557940         0   
4 -0.969840 -0.479571 -0.447597 -0.501292          0    -0.557940         0   

   pclass_2  pclass_3  sex_female  ...  embarked_Q  embarked_S  \
0         0         1           1  ...           1           0   
1         1         0           1  ...           0           0   
2         0         1           1  ...           0           1   
3         0         1           0  ...           0           0   
4         0         1           1  ...           0   

La réinitialisation des index a bien été effectuée. Cette opération ne modifie pas les données elles-mêmes, mais elle remplace les anciens numéros de lignes hérités du dataset initial par une numérotation simple à partir de `0`. Les fichiers exportés seront donc plus propres et plus lisibles pour les étapes suivantes.

Les dimensions finales montrent que le preprocessing aboutit à deux jeux de données explicatives cohérents : `X_train` contient **1047 observations** et **22 variables**, tandis que `X_test` contient **262 observations** avec exactement les mêmes **22 variables**. Les variables cibles `y_train` et `y_test` ont également été conservées séparément avec un nombre de lignes correspondant à leurs jeux respectifs.

L’aperçu de `X_train` confirme que le dataset est désormais entièrement numérique. Les variables quantitatives comme `age`, `sibsp`, `parch`, `fare` et `family_size` apparaissent sous forme standardisée, ce qui explique la présence de valeurs positives et négatives. À l’inverse, les variables issues de l’encodage catégoriel, comme `pclass_3`, `sex_female`, `title_grouped_Miss` ou `family_group_small_family`, restent codées en `0` et `1`, ce qui les rend directement exploitables par les futurs modèles.

Cette dernière étape confirme donc que le prétraitement est terminé. Les jeux `xtrain.csv`, `xtest.csv`, `ytrain.csv` et `ytest.csv` constituent désormais une base propre, cohérente et prête à être utilisée dans le notebook suivant consacré à la sélection des variables.

## Conclusion

Dans ce notebook, nous avons préparé le dataset Titanic pour les étapes suivantes du travail de machine learning. Le prétraitement a consisté à exploiter certaines informations issues de la phase 1 par du feature engineering, à supprimer les variables brutes devenues inutiles, à traiter les valeurs manquantes, à encoder les variables catégorielles et à standardiser les variables quantitatives pertinentes.

Une attention particulière a été portée à la structure familiale. En plus de la variable `family_size`, une variable catégorielle `family_group` a été construite afin de mieux refléter les différences observées dans la phase 1 entre les passagers voyageant seuls, en petite famille ou en grande famille. Cette transformation permet de mieux représenter une relation qui n’apparaissait pas strictement linéaire.

À l’issue de ces transformations, les jeux de données d’entraînement et de test sont entièrement exploitables pour la modélisation. Les fichiers `xtrain.csv`, `xtest.csv`, `ytrain.csv` et `ytest.csv` ont été sauvegardés et serviront de base au notebook suivant consacré à la sélection des variables.

La prochaine étape consistera à comparer plusieurs approches de sélection de variables : une sélection personnelle fondée sur l’analyse exploratoire, la méthode Variance Threshold et la méthode SelectKBest.